# Indian Residential Energy Consumption Pattern Analysis

**Project:** Indian Residential Energy Consumption Pattern Analysis Using Clustering and Classification

This notebook contains the reproducible data preparation, feature engineering, clustering, validation, and classification workflow used for the DWDM project.

> **Important:** The Streamlit application does not retrain the model for every user. This notebook is used to train the final model and save the fitted scaler and K-Means model. The application loads those saved `.pkl` files and uses them for prediction.


## 1. Project Pipeline

```text
Raw Smart-Meter CSV
        ↓
Data Cleaning & Coverage Filtering
        ↓
Consumer-Level Feature Engineering
        ↓
StandardScaler
        ↓
K-Means (K = 3)
        ↓
Consumer Clusters
        ↓
Saved Model + Scaler
        ↓
Streamlit Application
        ↓
New User CSV → Prediction → Profile → Recommendations
```


## 2. Imports and Configuration

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    adjusted_rand_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage

RANDOM_STATE = 42
BASE = Path("..")

DATA_FILE = BASE / "data" / "CEEW - Smart meter data Bareilly 2020.csv"
MODEL_DIR = BASE / "models"
RESULT_DIR = BASE / "final_model_results"
RESULTS_DIR = BASE / "results"

MODEL_DIR.mkdir(exist_ok=True)
RESULT_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

print("Data file:", DATA_FILE)
print("Model directory:", MODEL_DIR)


## 3. Load the Raw Smart-Meter Dataset

The dataset contains smart-meter readings with the main consumption variable `t_kWh` and timestamp information.


In [ ]:
df = pd.read_csv(DATA_FILE)

print("Rows:", len(df))
print("Columns:", list(df.columns))
display(df.head())


## 4. Basic Data Checks

In [ ]:
df["x_Timestamp"] = pd.to_datetime(df["x_Timestamp"], errors="coerce")
df["t_kWh"] = pd.to_numeric(df["t_kWh"], errors="coerce")

print("Missing values:")
display(df.isna().sum())

print("\nNumber of meters:", df["meter"].nunique())
print("Date range:", df["x_Timestamp"].min(), "to", df["x_Timestamp"].max())
print("Negative consumption values:", (df["t_kWh"] < 0).sum())


## 5. Coverage Filtering

For 2020, the expected number of 3-minute readings is:

`366 × 24 × 20 = 175,680 readings per meter`

A minimum coverage of **80%** is used for annual consumer-level analysis.


In [ ]:
EXPECTED_READINGS = 366 * 24 * 20
MIN_COVERAGE = 80

coverage = (
    df.groupby("meter").size()
    .rename("readings")
    .to_frame()
)

coverage["coverage_percent"] = (
    coverage["readings"] / EXPECTED_READINGS * 100
)

coverage["retained"] = coverage["coverage_percent"] >= MIN_COVERAGE

print("Meters before filtering:", len(coverage))
print("Meters retained:", int(coverage["retained"].sum()))
print("Meters removed:", int((~coverage["retained"]).sum()))

display(coverage.sort_values("coverage_percent"))


In [ ]:
retained_meters = coverage.index[coverage["retained"]]

df_filtered = df[df["meter"].isin(retained_meters)].copy()

print("Filtered rows:", len(df_filtered))
print("Filtered meters:", df_filtered["meter"].nunique())


## 6. Feature Engineering

The final K-Means model uses these 11 consumer-level features:

- mean consumption
- maximum consumption
- sample standard deviation
- median consumption
- peak-to-average ratio
- daytime average
- evening average
- night average
- weekday average
- weekend average
- weekday/weekend difference

The time windows match the final training pipeline:
- Daytime: 06:00–18:00
- Evening: 18:00–22:00
- Night: 22:00–06:00


In [ ]:
def build_features(raw):
    data = raw.copy()

    data["x_Timestamp"] = pd.to_datetime(data["x_Timestamp"], errors="coerce")
    data["t_kWh"] = pd.to_numeric(data["t_kWh"], errors="coerce")
    data = data.dropna(subset=["meter", "x_Timestamp", "t_kWh"])
    data = data[data["t_kWh"] >= 0].copy()

    data["hour"] = (
        data["x_Timestamp"].dt.hour
        + data["x_Timestamp"].dt.minute / 60.0
    )
    data["weekday"] = data["x_Timestamp"].dt.weekday

    rows = []

    for meter, g in data.groupby("meter"):
        c = g["t_kWh"]

        daytime = g[(g["hour"] >= 6) & (g["hour"] < 18)]
        evening = g[(g["hour"] >= 18) & (g["hour"] < 22)]
        night = g[(g["hour"] >= 22) | (g["hour"] < 6)]

        weekday = g[g["weekday"] < 5]
        weekend = g[g["weekday"] >= 5]

        mean_c = c.mean()

        rows.append({
            "meter": meter,
            "mean_consumption": mean_c,
            "max_consumption": c.max(),
            "std_consumption": c.std(),  # sample std, ddof=1
            "median_consumption": c.median(),
            "peak_to_average": c.max() / mean_c if mean_c else 0,
            "daytime_avg": daytime["t_kWh"].mean(),
            "evening_avg": evening["t_kWh"].mean(),
            "night_avg": night["t_kWh"].mean(),
            "weekday_avg": weekday["t_kWh"].mean(),
            "weekend_avg": weekend["t_kWh"].mean(),
            "weekday_weekend_difference": abs(
                weekday["t_kWh"].mean() - weekend["t_kWh"].mean()
            ),
        })

    return pd.DataFrame(rows)

features = build_features(df_filtered)

FEATURES = [
    "mean_consumption",
    "max_consumption",
    "std_consumption",
    "median_consumption",
    "peak_to_average",
    "daytime_avg",
    "evening_avg",
    "night_avg",
    "weekday_avg",
    "weekend_avg",
    "weekday_weekend_difference",
]

display(features)


In [ ]:
features.to_csv(
    BASE / "data" / "consumer_features_from_notebook.csv",
    index=False
)

X = features[FEATURES].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Feature matrix:", X.shape)
print("Scaled matrix:", X_scaled.shape)


## 7. Select the Number of Clusters

K-Means is evaluated for several values of K using the elbow curve, silhouette score, and Davies-Bouldin index.


In [ ]:
evaluation = []

for k in range(2, 8):
    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=10
    )
    labels = model.fit_predict(X_scaled)

    evaluation.append({
        "k": k,
        "inertia": model.inertia_,
        "silhouette": silhouette_score(X_scaled, labels),
        "davies_bouldin": davies_bouldin_score(X_scaled, labels),
    })

evaluation_df = pd.DataFrame(evaluation)
display(evaluation_df)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(evaluation_df["k"], evaluation_df["inertia"], marker="o")
ax.set_xlabel("Number of clusters (K)")
ax.set_ylabel("Inertia")
ax.set_title("Elbow Curve")
ax.grid(True, alpha=0.25)
plt.show()


## 8. Final K-Means Model

The final project model uses **K = 3** consumer groups.


In [ ]:
FINAL_K = 3

final_kmeans = KMeans(
    n_clusters=FINAL_K,
    random_state=RANDOM_STATE,
    n_init=10
)

labels = final_kmeans.fit_predict(X_scaled)
features["cluster"] = labels

print("Cluster sizes:")
display(features["cluster"].value_counts().sort_index())


In [ ]:
cluster_profiles = features.groupby("cluster")[FEATURES].mean()
display(cluster_profiles)


## 9. PCA Visualization

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "cluster": labels
})

print("Explained variance:")
print("PC1:", round(pca.explained_variance_ratio_[0] * 100, 2), "%")
print("PC2:", round(pca.explained_variance_ratio_[1] * 100, 2), "%")

for cluster in sorted(pca_df["cluster"].unique()):
    subset = pca_df[pca_df["cluster"] == cluster]
    plt.scatter(subset["PC1"], subset["PC2"], label=f"Cluster {cluster}")

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Consumer Clusters in PCA Space")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()


## 10. Compare HAC and GMM

These models are used as alternative clustering methods for comparison. They do not replace the final K-Means model used by the application.


In [ ]:
hac = AgglomerativeClustering(n_clusters=3, linkage="ward")
hac_labels = hac.fit_predict(X_scaled)

gmm = GaussianMixture(
    n_components=3,
    random_state=RANDOM_STATE
)
gmm_labels = gmm.fit_predict(X_scaled)

comparison = pd.DataFrame({
    "method": ["K-Means", "HAC", "GMM"],
    "silhouette": [
        silhouette_score(X_scaled, labels),
        silhouette_score(X_scaled, hac_labels),
        silhouette_score(X_scaled, gmm_labels),
    ],
    "davies_bouldin": [
        davies_bouldin_score(X_scaled, labels),
        davies_bouldin_score(X_scaled, hac_labels),
        davies_bouldin_score(X_scaled, gmm_labels),
    ],
})

display(comparison)

print("K-Means vs HAC ARI:",
      adjusted_rand_score(labels, hac_labels))
print("K-Means vs GMM ARI:",
      adjusted_rand_score(labels, gmm_labels))


## 11. Classification of the Discovered Cluster Labels

The cluster labels produced by K-Means are used as the target for classification experiments.

**Important:** these are not externally observed household classes. The classifiers are learning to reproduce the cluster structure discovered by K-Means.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    labels,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=labels
)

classifiers = {
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "SVM": SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE),
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ),
}

classification_results = []

for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)

    classification_results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, average="weighted", zero_division=0),
        "recall": recall_score(y_test, pred, average="weighted", zero_division=0),
        "f1": f1_score(y_test, pred, average="weighted", zero_division=0),
    })

classification_df = pd.DataFrame(classification_results)
display(classification_df)


## 12. Time-of-Day Consumer Profile

The four time shares are calculated separately for the user-facing profile:

- Morning: 06:00–12:00
- Afternoon: 12:00–18:00
- Evening: 18:00–22:00
- Night: 22:00–06:00

These shares are used by the application to describe when a consumer uses more energy and generate data-based recommendations.


In [ ]:
def build_time_profile(raw):
    data = raw.copy()
    data["x_Timestamp"] = pd.to_datetime(data["x_Timestamp"], errors="coerce")
    data["t_kWh"] = pd.to_numeric(data["t_kWh"], errors="coerce")
    data = data.dropna(subset=["meter", "x_Timestamp", "t_kWh"])
    data = data[data["t_kWh"] >= 0].copy()

    data["hour"] = (
        data["x_Timestamp"].dt.hour
        + data["x_Timestamp"].dt.minute / 60.0
    )

    periods = {
        "Morning": (6, 12),
        "Afternoon": (12, 18),
        "Evening": (18, 22),
        "Night": (22, 24),
    }

    result = []

    for meter, g in data.groupby("meter"):
        total = g["t_kWh"].sum()
        row = {"meter": meter}

        for name, (start, end) in periods.items():
            row[f"{name.lower()}_share"] = (
                g.loc[(g["hour"] >= start) & (g["hour"] < end), "t_kWh"].sum()
                / total if total else 0
            )

        row["night_share"] += (
            g.loc[g["hour"] < 6, "t_kWh"].sum() / total if total else 0
        )

        shares = {k: row[f"{k.lower()}_share"]
                  for k in ["Morning", "Afternoon", "Evening", "Night"]}
        row["dominant_period"] = max(shares, key=shares.get)

        result.append(row)

    return pd.DataFrame(result)

time_profile = build_time_profile(df_filtered)
display(time_profile)


## 13. Save Final Model Artifacts

These two files are the important connection between the training notebook and the Streamlit application.


In [ ]:
import joblib

joblib.dump(scaler, MODEL_DIR / "final_kmeans_scaler.pkl")
joblib.dump(final_kmeans, MODEL_DIR / "final_kmeans.pkl")

features.to_csv(RESULT_DIR / "household_profiles.csv", index=False)
cluster_profiles.to_csv(RESULT_DIR / "cluster_profiles.csv")
time_profile.to_csv(RESULT_DIR / "time_profiles.csv", index=False)

print("Saved:")
print(MODEL_DIR / "final_kmeans_scaler.pkl")
print(MODEL_DIR / "final_kmeans.pkl")


## 14. Verify the Saved Model

A new user's data must be transformed using the **same scaler fitted during training**. The app then passes the scaled features to the saved K-Means model.


In [ ]:
loaded_scaler = joblib.load(MODEL_DIR / "final_kmeans_scaler.pkl")
loaded_kmeans = joblib.load(MODEL_DIR / "final_kmeans.pkl")

test_scaled = loaded_scaler.transform(features[FEATURES])
test_predictions = loaded_kmeans.predict(test_scaled)

print("Predictions from saved model:")
print(test_predictions)
print("Same as training labels:",
      np.array_equal(test_predictions, labels))


## 15. Final Project Outputs

After running this notebook, the project contains:

```text
models/
├── final_kmeans.pkl
└── final_kmeans_scaler.pkl

final_model_results/
├── household_profiles.csv
├── cluster_profiles.csv
└── time_profiles.csv
```

The **Streamlit `app.py` uses the saved model files**. It does not need to rerun the full 6.6-million-row training process every time a user uploads a CSV.

### Reproducibility

If the dataset, feature definitions, and `random_state=42` are unchanged, running the training cells again reproduces the same K-Means training setup.


## 16. Production Training Step for the Streamlit Application

Run the following cell after the analysis cells when the final model needs to be regenerated. It overwrites the model artifacts used by `app.py` with the model trained from this notebook.

The application expects these exact files:

```text
models/final_kmeans.pkl
models/final_kmeans_scaler.pkl
```


In [ ]:
# FINAL PRODUCTION TRAINING CELL
# This cell is the authoritative training step for the Streamlit application.

import joblib

# Refit the scaler and K-Means from the current notebook data.
production_scaler = StandardScaler()
production_X_scaled = production_scaler.fit_transform(features[FEATURES])

production_kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)
production_labels = production_kmeans.fit_predict(production_X_scaled)

# Save the exact artifacts consumed by app.py.
joblib.dump(
    production_scaler,
    MODEL_DIR / "final_kmeans_scaler.pkl"
)
joblib.dump(
    production_kmeans,
    MODEL_DIR / "final_kmeans.pkl"
)

# Save the final household and cluster profiles.
production_profiles = features.copy()
production_profiles["cluster"] = production_labels

production_cluster_profiles = (
    production_profiles.groupby("cluster")[FEATURES].mean()
)

production_profiles.to_csv(
    RESULT_DIR / "household_profiles.csv",
    index=False
)
production_cluster_profiles.to_csv(
    RESULT_DIR / "cluster_profiles.csv"
)

print("Production model trained and saved.")
print("Scaler:", MODEL_DIR / "final_kmeans_scaler.pkl")
print("K-Means:", MODEL_DIR / "final_kmeans.pkl")
print()
print("Cluster sizes:")
print(production_profiles["cluster"].value_counts().sort_index())


In [ ]:
# FINAL ARTIFACT VERIFICATION
# Reload exactly what app.py will use and verify that it reproduces
# the labels generated during production training.

saved_scaler = joblib.load(MODEL_DIR / "final_kmeans_scaler.pkl")
saved_kmeans = joblib.load(MODEL_DIR / "final_kmeans.pkl")

reloaded_scaled = saved_scaler.transform(features[FEATURES])
reloaded_labels = saved_kmeans.predict(reloaded_scaled)

print("Saved model verification:")
print("Predictions match production labels:",
      np.array_equal(reloaded_labels, production_labels))

print("\nSaved-model cluster sizes:")
print(pd.Series(reloaded_labels).value_counts().sort_index())

assert np.array_equal(reloaded_labels, production_labels),     "Saved model verification failed."

print("\nVerification passed.")
